# Hardware Validation Walkthrough

This notebook reproduces the IBM Fez hardware validation results from the qb-compiler README.

## Setup

In [ ]:
# pip install qb-compiler

from qb_compiler import QBCompiler, QBCircuit, check_viability
from qb_compiler.config import BACKEND_CONFIGS
import json
from pathlib import Path

## Load calibration data

Load the IBM Fez March 2026 calibration snapshot used in our validation runs.

In [ ]:
# Path to calibration snapshot (from test fixtures)
cal_path = Path("../../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json")

if cal_path.exists():
    with open(cal_path) as f:
        cal_data = json.load(f)
    print(f"Loaded calibration: {cal_data.get('backend_name', 'ibm_fez')}")
    print(f"Timestamp: {cal_data.get('timestamp', 'unknown')}")
    print(f"Qubits: {cal_data.get('num_qubits', 'unknown')}")
else:
    print("Calibration snapshot not found — using default backend config")

## Preflight check — which circuits are viable?

Run `check_viability` on several circuits to see which are worth running on real hardware.

In [ ]:
def build_ghz(n):
    """Build a GHZ circuit on n qubits."""
    circuit = QBCircuit(n)
    circuit.h(0)
    for i in range(n - 1):
        circuit.cx(i, i + 1)
    circuit.measure_all()
    return circuit


circuits = {
    "GHZ-5": build_ghz(5),
    "GHZ-8": build_ghz(8),
    "GHZ-10": build_ghz(10),
}

for name, circ in circuits.items():
    result = check_viability(circ, backend="ibm_fez")
    print(f"\n{name}:")
    print(f"  Status: {result.status}")
    print(f"  Est. fidelity: {result.estimated_fidelity:.4f}")
    print(f"  Depth: {result.depth}")

Preflight correctly identifies GHZ circuits as VIABLE. Deeper circuits like QAOA-6 would show as MARGINAL, and QFT-6 as DO NOT RUN — saving QPU time.

## Compile with calibration-aware layout

Compile GHZ-8 using qb-compiler's CalibrationMapper, which scores layouts using post-routing 2Q gate counts and calibration data.

In [ ]:
compiler = QBCompiler.from_backend("ibm_fez")
ghz8 = build_ghz(8)

result = compiler.compile(ghz8)

print(f"Input depth:  {result.original_depth}")
print(f"Output depth: {result.compiled_depth}")
print(f"Reduction:    {result.depth_reduction_pct:.1f}%")
print(f"Fidelity:     {result.estimated_fidelity:.4f}")
print(f"Time:         {result.compilation_time_ms:.1f} ms")
print(f"\nQubit mapping: {result.qubit_mapping}")

## Compare with Qiskit default

Transpile the same circuit with Qiskit `optimization_level=3` and compare qubit selections.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.fake_provider import GenericBackendV2

# Build equivalent Qiskit circuit
qc = QuantumCircuit(8, 8)
qc.h(0)
for i in range(7):
    qc.cx(i, i + 1)
qc.measure(range(8), range(8))

# Transpile with Qiskit default
backend = GenericBackendV2(156)
qiskit_tc = transpile(qc, backend, optimization_level=3)

print(f"Qiskit layout:      {qiskit_tc.layout.initial_layout}")
print(f"Qiskit depth:       {qiskit_tc.depth()}")
print(f"Qiskit 2Q gates:    {qiskit_tc.count_ops().get('cx', 0) + qiskit_tc.count_ops().get('ecr', 0)}")
print()
print(f"qb-compiler layout: {result.qubit_mapping}")
print(f"qb-compiler depth:  {result.compiled_depth}")

## Hardware results

Measured fidelity from IBM Fez (March 2026), 4096 shots per circuit.

| Circuit | Qiskit opt_level=3 | qb-compiler | Delta |
|---------|-------------------|-------------|-------|
| GHZ-3   | 96.5%             | 96.7%       | +0.2% |
| GHZ-5   | 92.5%             | 93.2%       | +0.7% |
| GHZ-8   | 82.1%             | 87.5%       | **+5.3%** |
| GHZ-10  | 78.8%             | 79.8%       | +1.0% |

Fidelity = P(000...0) + P(111...1) over 4096 shots.

In [ ]:
# Load full results if available
results_path = Path("../../results/validation_20260314_143516.json")

if results_path.exists():
    with open(results_path) as f:
        validation = json.load(f)
    print("Full validation results loaded.")
    for entry in validation.get("results", []):
        name = entry.get("circuit", "")
        qiskit_f = entry.get("qiskit_fidelity", 0)
        qbc_f = entry.get("qbc_fidelity", 0)
        delta = qbc_f - qiskit_f
        print(f"  {name:8s}  Qiskit: {qiskit_f:.4f}  qb-compiler: {qbc_f:.4f}  delta: {delta:+.4f}")
else:
    print("Validation results file not found — see table above for measured numbers.")

## Preflight accuracy

How well did preflight predictions match measured hardware results?

| Circuit | Preflight Status | Predicted Fidelity | Measured Fidelity | Correct? |
|---------|-----------------|-------------------|-------------------|----------|
| GHZ-3   | VIABLE          | ~0.95              | 0.967             | Yes      |
| GHZ-5   | VIABLE          | ~0.90              | 0.932             | Yes      |
| GHZ-8   | VIABLE          | ~0.85              | 0.875             | Yes      |
| GHZ-10  | VIABLE          | ~0.80              | 0.798             | Yes      |
| QAOA-6  | MARGINAL        | ~0.18              | 0.059             | Yes      |
| QFT-6   | DO NOT RUN      | ~0.02              | 0.021             | Yes      |

qb-compiler correctly identified all non-viable circuits before execution.

## Cost estimation

Estimated cost for each circuit at 4096 shots on IBM Fez.

In [ ]:
compiler = QBCompiler.from_backend("ibm_fez")

test_circuits = {
    "GHZ-5": build_ghz(5),
    "GHZ-8": build_ghz(8),
    "GHZ-10": build_ghz(10),
}

print(f"{'Circuit':<10} {'Status':<14} {'Est. Fidelity':>14} {'Cost (4096 shots)':>18}")
print("-" * 60)

for name, circ in test_circuits.items():
    viability = check_viability(circ, backend="ibm_fez")
    compiled = compiler.compile(circ)
    cost = compiler.estimate_cost(compiled.compiled_circuit, shots=4096)
    print(
        f"{name:<10} {viability.status:<14} "
        f"{compiled.estimated_fidelity:>14.4f} "
        f"${cost.total_usd:>17.4f}"
    )

print()
print("QAOA-6 and QFT-6 would have wasted QPU time without preflight.")